In [ ]:
!pip install -q kaggle

In [ ]:
from google.colab import files
files.upload()  # Upload your kaggle.json here when prompted

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"sadanandukey","key":"1b172bea2377269349892e01dcc85d2b"}'}

In [ ]:
import os

# Move kaggle.json to correct directory
os.makedirs('/root/.kaggle', exist_ok=True)
!cp kaggle.json /root/.kaggle/
!chmod 600 /root/.kaggle/kaggle.json

# Create data directory
os.makedirs('/content/data', exist_ok=True)

# Download APTOS 2019 dataset
!kaggle competitions download -c aptos2019-blindness-detection -p /content/data

# Unzip
!unzip -q /content/data/aptos2019-blindness-detection.zip -d /content/data/

# Verify files exist
print("Files in /content/data/:")
print(os.listdir('/content/data'))

100% 9.51G/9.51G [00:50<00:00, 201MB/s]

Files in /content/data/:
['train_images', 'aptos2019-blindness-detection.zip', 'test_images', 'train.csv', 'sample_submission.csv', 'test.csv']


In [ ]:
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.models as models
from sklearn.metrics import roc_auc_score, confusion_matrix, classification_report

IMG_SIZE = 380
BATCH_SIZE = 16        # Reduced to avoid memory issues on free Colab GPU
EPOCHS = 5
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using device:", DEVICE)

Using device: cuda


In [ ]:
df = pd.read_csv('/content/data/train.csv')
print("Columns:", df.columns.tolist())
print("Shape:", df.shape)
print(df.head())

# Rename columns if needed
df.columns = ['id_code', 'diagnosis']

# Binary label: 0 = No/Mild DR, 1 = Referable DR (moderate or worse)
df['label'] = (df['diagnosis'] >= 2).astype(int)
print("\nLabel distribution:")
print(df['label'].value_counts())

Columns: ['id_code', 'diagnosis']
Shape: (3662, 2)
        id_code  diagnosis
0  000c1434d8d7          2
1  001639a390f0          4
2  0024cdab0c1e          1
3  002c21358ce6          0
4  005b95c28852          0

Label distribution:
label
0    2175
1    1487
Name: count, dtype: int64


In [ ]:
class RetinaDataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_name = self.df.iloc[idx]['id_code'] + '.png'
        img_path = os.path.join(self.img_dir, img_name)
        image = cv2.imread(img_path)
        if image is None:
            # fallback: return blank image if file missing
            image = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
        else:
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        label = self.df.iloc[idx]['label']
        if self.transform:
            image = self.transform(image)
        return image, label

train_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
])

val_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
])

IMG_DIR = '/content/data/train_images'

train_df, val_df = train_test_split(df, test_size=0.2, stratify=df['label'], random_state=42)
train_dataset = RetinaDataset(train_df, IMG_DIR, train_transform)
val_dataset   = RetinaDataset(val_df,   IMG_DIR, val_transform)
train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader    = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Train size: {len(train_dataset)}, Val size: {len(val_dataset)}")

Train size: 2929, Val size: 733


In [ ]:
import warnings
warnings.filterwarnings("ignore")

model = models.efficientnet_b4(weights=models.EfficientNet_B4_Weights.DEFAULT)
num_ftrs = model.classifier[1].in_features
model.classifier[1] = nn.Linear(num_ftrs, 1)   # Binary output
model = model.to(DEVICE)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)
print("Model ready ✅")

Downloading: "https://download.pytorch.org/models/efficientnet_b4_rwightman-23ab8bcd.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b4_rwightman-23ab8bcd.pth


100%|██████████| 74.5M/74.5M [00:00<00:00, 131MB/s]


Model ready ✅


In [ ]:
best_acc = 0.0

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images = images.to(DEVICE)
        labels = labels.float().to(DEVICE).unsqueeze(1)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    # Validation
    model.eval()
    all_preds, all_labels_val = [], []
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(DEVICE)
            labels = labels.float().to(DEVICE).unsqueeze(1)
            outputs = torch.sigmoid(model(images))
            preds   = (outputs > 0.5).float()
            all_preds.extend(preds.cpu().numpy())
            all_labels_val.extend(labels.cpu().numpy())

    acc = (np.array(all_preds) == np.array(all_labels_val)).mean()
    print(f'Epoch {epoch+1}/{EPOCHS} | Loss: {running_loss:.4f} | Val Accuracy: {acc:.4f}')

    if acc > best_acc:
        best_acc = acc
        torch.save(model.state_dict(), 'best_model.pth')
        print(f'  ✅ Best model saved (acc={best_acc:.4f})')

Epoch 1/5 | Loss: 75.0131 | Val Accuracy: 0.9345
  ✅ Best model saved (acc=0.9345)
Epoch 2/5 | Loss: 40.6273 | Val Accuracy: 0.9277
Epoch 3/5 | Loss: 33.9754 | Val Accuracy: 0.9386
  ✅ Best model saved (acc=0.9386)
Epoch 4/5 | Loss: 29.2780 | Val Accuracy: 0.9263
Epoch 5/5 | Loss: 28.9629 | Val Accuracy: 0.9236


In [ ]:
model.load_state_dict(torch.load('best_model.pth'))
model.eval()

all_probs, all_true = [], []
with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(DEVICE)
        labels = labels.float().to(DEVICE).unsqueeze(1)
        outputs = torch.sigmoid(model(images))
        all_probs.extend(outputs.cpu().numpy())
        all_true.extend(labels.cpu().numpy())

all_probs = np.array(all_probs).flatten()
all_true  = np.array(all_true).flatten()
preds     = (all_probs > 0.5).astype(int)

tn, fp, fn, tp = confusion_matrix(all_true, preds).ravel()
sensitivity = tp / (tp + fn)
specificity = tn / (tn + fp)
auc         = roc_auc_score(all_true, all_probs)

print(f"\n===== Final Results =====")
print(f"Sensitivity (Recall) : {sensitivity:.4f}")
print(f"Specificity          : {specificity:.4f}")
print(f"AUC-ROC              : {auc:.4f}")
print("\n", classification_report(all_true, preds, target_names=['No/mild DR', 'Referable DR']))


===== Final Results =====
Sensitivity (Recall) : 0.9362
Specificity          : 0.9402
AUC-ROC              : 0.9818

               precision    recall  f1-score   support

  No/mild DR       0.96      0.94      0.95       435
Referable DR       0.91      0.94      0.93       298

    accuracy                           0.94       733
   macro avg       0.94      0.94      0.94       733
weighted avg       0.94      0.94      0.94       733

